# picoNewton_v4

## Anisotropic Womersley forcing, vector-resolved membrane mechanics, and Piezo1 mechanosensory response

This notebook is the primary executable interface for `picoNewton_v4`. It is designed to run from top to bottom on the Google Colab free tier and to function as a complete scientific and computational document.

The notebook will:

1. mount Google Drive;
2. create a persistent main runtime environment;
3. create a new UTC date-time-stamped run directory for every execution;
4. clone and install `picoNewton_v4` automatically from GitHub;
5. record the exact Git commit and Python environment;
6. run the package verification and test suite;
7. execute all six arterial cases;
8. generate tables, waveforms, figures, validation records, and checksums;
9. copy the complete run to Google Drive without overwriting earlier runs.

The default `quick` profile is suitable for routine Colab verification. The `full` profile uses the complete declared numerical resolution.

## Scientific question

The model tests whether anisotropic near-wall hydrodynamic forcing produces mechanosensory information that is distinct from wall shear stress.

The six arterial cases are:

- Aortic Root
- Thoracic Aorta
- Femoral
- Carotid
- Iliac
- Brachial

The hydrodynamic ground truth is based on the six-artery anisotropic Womersley model associated with DOI `10.1038/s41598-026-47474-x`.

The calculation preserves three physically different observables:

### Wall shear stress

\[
\tau_w(t)=\mu\left.\frac{\partial u_z}{\partial r}\right|_{r=R},
\]

with units of Pa.

### Signed transverse Lamb force

\[
F_{\mathrm{signed}}(t)=A_{EC}\int_{R-\delta_{EC}}^{R} f_r(r,t)\,dr,
\]

with units of N. This observable preserves normal-load direction.

### Nonnegative Lamb-force exposure

\[
F_{\mathrm{exposure}}(t)=A_{EC}\int_{R-\delta_{EC}}^{R}|f_r(r,t)|\,dr,
\]

with units of N. This observable represents magnitude-sensitive exposure and is never used as a signed load.

## Vector-resolved membrane and Piezo1 model

Tangential and normal forcing are passed through separate passive generalized standard-linear-solid branches. Fast and slow branches preserve short- and long-timescale mechanics.

The model retains:

- tangential strain;
- signed normal strain;
- nonnegative exposure strain;
- apical and junctional membrane tension;
- local equivalent pressure;
- force-vector angle.

Local tension is mapped to equivalent pressure through the configured curvature radius:

\[
P_{eq}(t)=\frac{2T(t)}{r_{eff}}.
\]

The four-state Piezo1 model then produces:

\[
P_{Open}(t),
\]

\[
I(t)=N_{ch}g_{ch}P_{Open}(t)(V_m-E_{rev}),
\]

and a reduced-order calcium-scale endpoint.

The calcium endpoint is retained as a screening variable. It does not replace a complete endothelial calcium-handling model.

## Comparison matrix and hypotheses

The workflow evaluates:

- zero-load controls;
- WSS-only, signed-force-only, exposure-only, and combined vector pathways;
- RMS-, peak-, and work-matched controls;
- anisotropic versus isotropic responses;
- full harmonics versus harmonics 1–2;
- viscoelastic versus elastic interfaces;
- leave-one-artery-out causal WSS surrogates;
- multifeature artery-response distances.

The hypothesis families are:

- **H1:** hydrodynamic nonredundancy;
- **H2:** membrane-state nonredundancy;
- **H3a:** direct mechanosensory distinction from WSS;
- **H3b:** mechanosensory information not reproduced by a held-out WSS surrogate;
- **H4:** anisotropy-specific response;
- **H5:** higher-harmonic response;
- **H6:** artery-specific response features;
- **H7:** robustness across viscoelastic and elastic mechanical limits.

Effect sizes are always exported. A pass/fail table is produced only from explicitly configured current and calcium detection thresholds.

## Colab execution contract

This notebook uses a standard CPU runtime and limits thread counts to protect the free tier.

Persistent Drive root:

```text
MyDrive/picoNewton_v4_runtime/
```

Every execution creates a unique directory:

```text
MyDrive/picoNewton_v4_runtime/runs/YYYYMMDD_HHMMSS_UTC/
```

Each run contains:

```text
configs/       exact configuration files used
inputs/        artery and model inputs
logs/          installation, verification, and test logs
checkpoints/   stage-completion metadata
tables/        CSV result tables
figures/       generated PNG figures
validation/    validation and decision records
results/       compressed waveform arrays
governance/    interpretation boundaries and threshold metadata
provenance/    Git commit, hashes, and run metadata
environment/   Python and package versions
```

Previous runs are never overwritten.

In [ ]:
# Mount Google Drive.
from google.colab import drive

drive.mount('/content/drive')


## User configuration

Edit only this cell before execution.

- `REPO_REF` may be a branch or tag containing `picoNewton_v4/`.
- `PROFILE="quick"` is the free-tier default.
- `PROFILE="full"` selects the complete numerical resolution.
- The optional sensitivity scan is disabled by default because it is substantially more expensive.
- `CALIBRATION_RELATIVE_PATH` identifies the calibration file inside the package.

In [ ]:
# User configuration
REPO_URL = 'https://github.com/khalid-saqr/picoNewton.git'
REPO_REF = 'main'  # Change to the branch or tag that contains picoNewton_v4.
PACKAGE_SUBDIR = 'picoNewton_v4'

PROFILE = 'quick'  # 'quick' or 'full'
RUN_SENSITIVITY_SCAN = False
CALIBRATION_RELATIVE_PATH = 'configs/literature_calibration.json'
MINIMUM_PASSING_ARTERIES = 4

assert PROFILE in {'quick', 'full'}


## Create the persistent and local runtime environments

Google Drive stores the durable run. `/content` is used for computation because local Colab storage is substantially faster than Drive for temporary numerical files.

The run identifier uses UTC and includes seconds plus a random suffix, preventing collisions even when two executions start within the same second.

In [ ]:
from __future__ import annotations

import os
import platform
import secrets
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

# Conservative CPU-thread settings for the Colab free tier.
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['NUMEXPR_NUM_THREADS'] = '2'

run_stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
RUN_ID = f'{run_stamp}_{secrets.token_hex(2)}'

DRIVE_MAIN = Path('/content/drive/MyDrive/picoNewton_v4_runtime')
RUN_DIR = DRIVE_MAIN / 'runs' / RUN_ID
LOCAL_ROOT = Path('/content/picoNewton_v4_work') / RUN_ID
REPO_DIR = LOCAL_ROOT / 'repository'
LOCAL_OUTPUT = LOCAL_ROOT / 'outputs'

RUN_SUBDIRS = {
    name: RUN_DIR / name
    for name in (
        'configs', 'inputs', 'logs', 'checkpoints', 'tables', 'figures',
        'validation', 'results', 'governance', 'provenance', 'environment'
    )
}

for path in [DRIVE_MAIN, DRIVE_MAIN / 'runs', RUN_DIR, LOCAL_ROOT, *RUN_SUBDIRS.values()]:
    path.mkdir(parents=True, exist_ok=True)

print('Run ID:', RUN_ID)
print('Drive run directory:', RUN_DIR)
print('Local work directory:', LOCAL_ROOT)


## Clone and install from GitHub

The notebook clones the selected GitHub ref into the local Colab workspace and installs the package in editable mode. The resolved Git commit is recorded before any model calculation.

The package is expected at:

```text
<repository>/picoNewton_v4/
```

The clone and installation logs are written to the timestamped Drive run.

In [ ]:
def run_logged(command, log_path, *, cwd=None):
    command = [str(item) for item in command]
    completed = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    Path(log_path).write_text(
        '$ ' + ' '.join(command) + '\n\n' + completed.stdout,
        encoding='utf-8',
    )
    print(completed.stdout[-4000:])
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with code {completed.returncode}: {command}')
    return completed.stdout

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run_logged(
    ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, REPO_DIR],
    RUN_SUBDIRS['logs'] / 'git_clone.log',
)

PACKAGE_DIR = REPO_DIR / PACKAGE_SUBDIR
if not PACKAGE_DIR.exists():
    raise FileNotFoundError(
        f'Package directory not found: {PACKAGE_DIR}. '
        'Check REPO_REF and PACKAGE_SUBDIR.'
    )

GIT_SHA = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True
).strip()

run_logged(
    [sys.executable, '-m', 'pip', 'install', '-e', f'{PACKAGE_DIR}[dev]'],
    RUN_SUBDIRS['logs'] / 'pip_install.log',
)

(RUN_SUBDIRS['provenance'] / 'git_commit.txt').write_text(GIT_SHA + '\n', encoding='utf-8')
(RUN_SUBDIRS['provenance'] / 'git_ref_requested.txt').write_text(REPO_REF + '\n', encoding='utf-8')
(RUN_SUBDIRS['provenance'] / 'repository_url.txt').write_text(REPO_URL + '\n', encoding='utf-8')

print('Package directory:', PACKAGE_DIR)
print('Resolved Git commit:', GIT_SHA)


## Verify the package and test suite

This stage performs three checks before the scientific workflow starts:

1. static package and notebook integrity;
2. import and version verification;
3. the complete package test suite.

The model run does not begin unless all checks pass.

In [ ]:
import json
import piconewton_v4

run_logged(
    [sys.executable, str(PACKAGE_DIR / 'scripts' / 'verify_package.py')],
    RUN_SUBDIRS['logs'] / 'package_verification.log',
    cwd=PACKAGE_DIR,
)
run_logged(
    [sys.executable, '-m', 'pytest', '-q'],
    RUN_SUBDIRS['logs'] / 'pytest.log',
    cwd=PACKAGE_DIR,
)

pip_freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_SUBDIRS['environment'] / 'pip_freeze.txt').write_text(pip_freeze, encoding='utf-8')

system_info = {
    'python': sys.version,
    'platform': platform.platform(),
    'processor': platform.processor(),
    'package_version': piconewton_v4.__version__,
    'git_sha': GIT_SHA,
    'profile': PROFILE,
}
(RUN_SUBDIRS['environment'] / 'system.json').write_text(
    json.dumps(system_info, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)

print(json.dumps(system_info, indent=2))


## Freeze inputs and calibration

The notebook copies the exact model configuration, calibration file, and six-artery input table into the Drive run before execution.

The supplied literature calibration is fully source-annotated, but several values remain cross-cell or model proxies. The notebook records this status without changing it.

In [ ]:
from piconewton_v4.calibration import load_parameterization

CALIBRATION_PATH = PACKAGE_DIR / CALIBRATION_RELATIVE_PATH
MODEL_CONFIG_PATH = PACKAGE_DIR / 'configs' / 'model.json'
ARTERY_INPUT_PATH = PACKAGE_DIR / 'data' / 'ground_truth_arteries.csv'

for source, destination in (
    (CALIBRATION_PATH, RUN_SUBDIRS['configs'] / CALIBRATION_PATH.name),
    (MODEL_CONFIG_PATH, RUN_SUBDIRS['configs'] / MODEL_CONFIG_PATH.name),
    (ARTERY_INPUT_PATH, RUN_SUBDIRS['inputs'] / ARTERY_INPUT_PATH.name),
):
    shutil.copy2(source, destination)

interface_parameters, endpoint_parameters, calibration_audit = load_parameterization(
    CALIBRATION_PATH
)

(RUN_SUBDIRS['governance'] / 'calibration_audit.json').write_text(
    json.dumps(calibration_audit, indent=2, sort_keys=True, default=str) + '\n',
    encoding='utf-8',
)

print('Calibration status:', calibration_audit['calibration_status'])
print('Missing source groups:', calibration_audit['missing_source_groups'])
print('Current threshold [pA]:', endpoint_parameters.current_detection_limit_pa)
print('Calcium threshold [nM]:', endpoint_parameters.calcium_detection_limit_nm)


## Execute the six-artery workflow

Numerical profiles:

| Profile | Radial order | Time points | Near-wall nodes |
|---|---:|---:|---:|
| `quick` | 48 | 256 | 48 |
| `full` | 150 | 2048 | 256 |

The workflow computes all hydrodynamic signals directly from the packaged artery inputs unless an explicit compatible hydrodynamic artifact is supplied.

The optional sensitivity scan is controlled by `RUN_SENSITIVITY_SCAN` and is disabled by default.

In [ ]:
from piconewton_v4.workflow import run_workflow

manifest = run_workflow(
    package_root=PACKAGE_DIR,
    output_root=LOCAL_OUTPUT,
    profile=PROFILE,
    calibration_path=CALIBRATION_PATH,
    run_scan=RUN_SENSITIVITY_SCAN,
)

(RUN_SUBDIRS['checkpoints'] / 'workflow_complete.json').write_text(
    json.dumps(
        {
            'run_id': RUN_ID,
            'git_sha': GIT_SHA,
            'profile': PROFILE,
            'status': manifest['status'],
        },
        indent=2,
        sort_keys=True,
    ) + '\n',
    encoding='utf-8',
)

print('Workflow status:', manifest['status'])
print('Local output:', LOCAL_OUTPUT)


## Validate, classify, and generate figures

The validation stage confirms:

- all six arteries are present;
- probability conservation and periodic closure;
- passive normal and tangential mechanics;
- separation of signed and exposure force classes;
- exposure is never used as a signed load;
- required output files are complete.

The decision table uses the current and calcium detection thresholds stored in the selected calibration file. These thresholds are written to the run provenance.

In [ ]:
import pandas as pd

from piconewton_v4.hypotheses import DecisionThresholds, write_decisions
from piconewton_v4.reporting import generate_standard_figures
from piconewton_v4.validation import validate_output_directory

validation_report = validate_output_directory(LOCAL_OUTPUT)
thresholds = DecisionThresholds(
    current_rms_pa=endpoint_parameters.current_detection_limit_pa,
    calcium_rms_nm=endpoint_parameters.calcium_detection_limit_nm,
    minimum_arteries=MINIMUM_PASSING_ARTERIES,
)

decisions = write_decisions(
    LOCAL_OUTPUT / 'hypothesis_effects.csv',
    LOCAL_OUTPUT / 'hypothesis_decisions.csv',
    thresholds,
    LOCAL_OUTPUT / 'decision_thresholds.json',
)
figure_paths = generate_standard_figures(LOCAL_OUTPUT, LOCAL_OUTPUT / 'figures')

(LOCAL_OUTPUT / 'output_validation.json').write_text(
    json.dumps(validation_report, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)

assert validation_report['passed'], validation_report
print(json.dumps(validation_report, indent=2))
print('Figures generated:', len(figure_paths))


## Inspect the generated results

This cell displays the principal tables and the six-artery current comparison. The complete numerical arrays remain in `waveforms.npz`.

Interpretation must distinguish:

- effect size from threshold classification;
- current from reduced-order calcium response;
- direct pathway differences from WSS-surrogate irreducibility;
- cell-scale force exposure from signed local normal loading;
- screening calibration from direct arterial-endothelial measurement.

In [ ]:
from IPython.display import display, Image

summary = pd.read_csv(LOCAL_OUTPUT / 'six_artery_summary.csv')
effects = pd.read_csv(LOCAL_OUTPUT / 'hypothesis_effects.csv')
surrogates = pd.read_csv(LOCAL_OUTPUT / 'loao_vector_surrogates.csv')

print('Six-artery summary')
display(summary.head(24))
print('Hypothesis decisions')
display(decisions)
print('Held-out WSS surrogates')
display(surrogates)

overview_figure = LOCAL_OUTPUT / 'figures' / 'six_artery_current_rms.png'
if overview_figure.exists():
    display(Image(filename=str(overview_figure)))


## Copy all outputs to Google Drive and generate checksums

The final stage sorts outputs into the timestamped Drive directory, writes run metadata, and creates SHA-256 checksums for every durable artifact.

The local `/content` workspace may disappear after the Colab session ends. The Drive run remains available.

In [ ]:
import hashlib

# Copy output files into organized Drive directories.
for path in LOCAL_OUTPUT.iterdir():
    if path.is_dir() and path.name == 'figures':
        for figure in path.glob('*.png'):
            shutil.copy2(figure, RUN_SUBDIRS['figures'] / figure.name)
    elif path.suffix == '.csv':
        shutil.copy2(path, RUN_SUBDIRS['tables'] / path.name)
    elif path.suffix == '.npz':
        shutil.copy2(path, RUN_SUBDIRS['results'] / path.name)
    elif path.name in {'validation.json', 'output_validation.json', 'decision_thresholds.json'}:
        shutil.copy2(path, RUN_SUBDIRS['validation'] / path.name)
    elif path.name == 'manifest.json':
        shutil.copy2(path, RUN_SUBDIRS['provenance'] / path.name)
    else:
        shutil.copy2(path, RUN_SUBDIRS['results'] / path.name)

run_metadata = {
    'run_id': RUN_ID,
    'started_utc': run_stamp,
    'completed_utc': datetime.now(timezone.utc).isoformat(),
    'repository_url': REPO_URL,
    'requested_ref': REPO_REF,
    'resolved_git_sha': GIT_SHA,
    'package_subdirectory': PACKAGE_SUBDIR,
    'profile': PROFILE,
    'sensitivity_scan': RUN_SENSITIVITY_SCAN,
    'calibration_file': CALIBRATION_RELATIVE_PATH,
    'drive_run_directory': str(RUN_DIR),
    'validation_passed': validation_report['passed'],
}
(RUN_SUBDIRS['provenance'] / 'run_metadata.json').write_text(
    json.dumps(run_metadata, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)

checksums = {}
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file() and path.name != 'SHA256SUMS.json':
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        checksums[path.relative_to(RUN_DIR).as_posix()] = {
            'sha256': digest,
            'size_bytes': path.stat().st_size,
        }
(RUN_SUBDIRS['provenance'] / 'SHA256SUMS.json').write_text(
    json.dumps(checksums, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)

print('Completed run:', RUN_ID)
print('Persistent output:', RUN_DIR)
print('Files checksummed:', len(checksums))


# Run complete

The timestamped Google Drive directory contains the complete computational record for this execution.

A future run will create a new directory and will not modify this one. To reproduce this exact run, use the recorded Git SHA, configuration files, artery inputs, Python environment, and checksum manifest stored under `provenance/` and `environment/`.